# US Structure Data Pipeline

Build city-level structure polygons for U.S. cities by combining:

- **Overture Maps buildings** for primary polygons and building attributes.
- **Microsoft Global ML Building Footprints** as a polygon gap-filler and height/confidence source.
- **OSMnx / OpenStreetMap** for building tags, names, floors, units, and heights where mapped.
- **USACE National Structure Inventory (NSI)** for occupancy type, stories, residential units, population proxies, employees, students, values, and Census block IDs.
- **Census ACS** for a city-level average household-size fallback when structure-level occupant counts are not available.

The output is one row per structure polygon with raw source fields collapsed into auditable columns such as `StructureTypeSource`, `NumUnitsSource`, `NumStoriesSource`, and `OccupantCountMethod`.

## Install Dependencies

Run this once if the environment is missing packages. These are ordinary Python/geospatial packages; no AI model packages are used.

```python
%pip install -r requirements.txt
```

In [1]:
from structure_pipeline import PipelineConfig, build_many_cities

## Configure Cities and Sources

Add as many cities as needed. For large cities, the first run can take time because OSM, Overture, Microsoft, NSI, and Census calls are network-bound. Local source files are reused on later runs.

In [2]:
CITIES = [
    {"city": "Houston", "state": "Texas"},
    # {"city": "Austin", "state": "Texas"},
    # {"city": "Seattle", "state": "Washington"},
]

config = PipelineConfig(
    data_dir="data",
    output_dir="data/output",
    raw_dir="data/raw",
    cache_dir="cache",
    country="USA",
    download_missing=True,
    use_overture=True,
    use_microsoft=True,
    use_osm=True,
    use_nsi=True,
    use_census=True,
    add_microsoft_unmatched=True,
    # Smaller values reduce NSI response memory per request but increase request count.
    nsi_tile_size_deg=0.08,
)

## Run Pipeline

Outputs are written to `data/output/{city}_{state}_usa_structures.parquet` and a combined `data/output/structures_master.parquet`.

In [3]:
structures = build_many_cities(CITIES, config)
structures.head()


=== Houston, Texas ===
Overture polygons: 615,237
Microsoft polygons: 559,919
Merged footprint polygons: 626,555
OSM source skipped: Expecting value: line 1 column 1 (char 0)
OSM building polygons: 0
NSI tile skipped: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
NSI tile skipped: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
NSI structure points: 634,745
Saved 626,555 rows to data/output/houston_texas_usa_structures.parquet
Saved combined output to data/output/structures_master.parquet


,StructureID,City,State,Country,FootprintSource,OvertureID,MicrosoftID,OSMID,NSI_FD_ID,NSI_RecordCount,...,NSI_Pop2PM,NSI_EmpNum,NSI_Students,CBFIPS,FootprintArea_m2,Confidence_MS,HasParts,CensusAvgHouseholdSize,CensusSource,geometry
0,ovt_9c8bec8f-6749-4bac-9b11-b452ff1f40ba,Houston,Texas,USA,overture,9c8bec8f-6749-4bac-9b11-b452ff1f40ba,<NA>,<NA>,485468055.0,1.0,...,2.0,NaN,0.0,481576707001035,227.700179,NaN,False,2.46,ACS 2024 B25010_001E,"POLYGON ((-95.51719 29.57233, -95.51709 29.572..."
1,ovt_880f837b-e9ee-4011-88b2-b2ce9248e631,Houston,Texas,USA,overture,880f837b-e9ee-4011-88b2-b2ce9248e631,<NA>,<NA>,485467946.0,1.0,...,1.0,NaN,0.0,481576707001035,202.951426,NaN,False,2.46,ACS 2024 B25010_001E,"POLYGON ((-95.51798 29.57232, -95.51782 29.572..."
2,ovt_463c2394-544e-4d56-af2c-17916718c323,Houston,Texas,USA,overture,463c2394-544e-4d56-af2c-17916718c323,<NA>,<NA>,485468050.0,1.0,...,0.0,NaN,0.0,481576707001035,205.880461,NaN,False,2.46,ACS 2024 B25010_001E,"POLYGON ((-95.51917 29.57226, -95.51917 29.572..."
3,ovt_2557e107-93c7-48d2-97a9-da4150aba7a3,Houston,Texas,USA,overture,2557e107-93c7-48d2-97a9-da4150aba7a3,<NA>,<NA>,485468053.0,1.0,...,2.0,NaN,0.0,481576707001035,186.033304,NaN,False,2.46,ACS 2024 B25010_001E,"POLYGON ((-95.51894 29.57212, -95.51894 29.572..."
4,ovt_e77e6331-1dc8-468f-b813-03367ab2c121,Houston,Texas,USA,overture,e77e6331-1dc8-468f-b813-03367ab2c121,<NA>,<NA>,485468047.0,1.0,...,1.0,NaN,0.0,481576707001035,196.986313,NaN,False,2.46,ACS 2024 B25010_001E,"POLYGON ((-95.51808 29.57219, -95.51818 29.572..."


## Inspect Coverage

These checks show how often the final fields came from each source or inference method.

In [4]:
print("Rows:", len(structures))
print("Cities:", structures[["City", "State"]].drop_duplicates().to_dict("records"))

summary_cols = [
    "FootprintSource",
    "StructureTypeSource",
    "NumUnitsSource",
    "NumStoriesSource",
    "HeightSource",
    "OccupantCountMethod",
]
for col in summary_cols:
    print(f"\n{col}")
    print(structures[col].value_counts(dropna=False).head(20))

Rows: 626555
Cities: [{'City': 'Houston', 'State': 'Texas'}]

FootprintSource
FootprintSource
overture     615237
microsoft     11318
Name: count, dtype: int64

StructureTypeSource
StructureTypeSource
nsi_occtype         451896
<NA>                125372
overture_class       47562
overture_subtype      1725
Name: count, dtype: int64

NumUnitsSource
NumUnitsSource
inferred_single_family    397700
<NA>                      228855
Name: count, dtype: int64

NumStoriesSource
NumStoriesSource
nsi                482915
height_estimate    113370
<NA>                26549
overture             3721
Name: count, dtype: int64

HeightSource
HeightSource
overture     573222
<NA>          43150
microsoft     10183
Name: count, dtype: int64

OccupantCountMethod
OccupantCountMethod
nsi_max_2am_2pm_emp_students         485838
NaN                                  136097
num_units_x_census_household_size      4620
Name: count, dtype: int64


## Output Schema

Key normalized columns:

- `geometry`: structure polygon in EPSG:4326.
- `StructureType`: normalized type such as `residential`, `condo`, `apartment`, `commercial`, `hotel`, `garage`, `barn`, `industrial`, `warehouse`, `education`, `healthcare`, or `unknown`.
- `StructureTypeRaw` and `StructureTypeSource`: raw value and source used to derive `StructureType`.
- `NumUnits` and `NumUnitsSource`: explicit OSM/NSI residential units, or `inferred_single_family` when the source type clearly describes a single-family structure.
- `NumStories` and `NumStoriesSource`: OSM floors, Overture floors, NSI stories, or a height-derived estimate.
- `OccupantCount` and `OccupantCountMethod`: NSI population/employee/student proxy when present, otherwise `NumUnits * CensusAvgHouseholdSize` for residential structures.
- `CBFIPS`: Census block FIPS from NSI where available.

Occupant counts should be treated as estimates unless your downstream workflow has a stronger authoritative source.

In [6]:
columns = [
    "StructureID", "City", "State", "FootprintSource", "StructureType",
    "StructureTypeRaw", "StructureTypeSource", "NumUnits", "NumUnitsSource",
    "NumStories", "NumStoriesSource", "OccupantCount", "OccupantCountMethod",
    "CBFIPS", "FootprintArea_m2", "geometry",
]
structures[columns].head(100)

,StructureID,City,State,FootprintSource,StructureType,StructureTypeRaw,StructureTypeSource,NumUnits,NumUnitsSource,NumStories,NumStoriesSource,OccupantCount,OccupantCountMethod,CBFIPS,FootprintArea_m2,geometry
0,ovt_9c8bec8f-6749-4bac-9b11-b452ff1f40ba,Houston,Texas,overture,residential,RES1-1SNB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,3.0,nsi_max_2am_2pm_emp_students,481576707001035,227.700179,"POLYGON ((-95.51719 29.57233, -95.51709 29.572..."
1,ovt_880f837b-e9ee-4011-88b2-b2ce9248e631,Houston,Texas,overture,residential,RES1-1SNB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,1.0,nsi_max_2am_2pm_emp_students,481576707001035,202.951426,"POLYGON ((-95.51798 29.57232, -95.51782 29.572..."
2,ovt_463c2394-544e-4d56-af2c-17916718c323,Houston,Texas,overture,residential,RES1-1SNB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,1.0,nsi_max_2am_2pm_emp_students,481576707001035,205.880461,"POLYGON ((-95.51917 29.57226, -95.51917 29.572..."
3,ovt_2557e107-93c7-48d2-97a9-da4150aba7a3,Houston,Texas,overture,residential,RES1-1SNB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,3.0,nsi_max_2am_2pm_emp_students,481576707001035,186.033304,"POLYGON ((-95.51894 29.57212, -95.51894 29.572..."
4,ovt_e77e6331-1dc8-468f-b813-03367ab2c121,Houston,Texas,overture,residential,RES1-1SWB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,1.0,nsi_max_2am_2pm_emp_students,481576707001035,196.986313,"POLYGON ((-95.51808 29.57219, -95.51818 29.572..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,ovt_94210c9a-f771-46f4-a231-e4258179bc15,Houston,Texas,overture,residential,RES1-1SNB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,2.0,nsi_max_2am_2pm_emp_students,481576706021005,200.810591,"POLYGON ((-95.50375 29.59592, -95.50376 29.595..."
96,ovt_bbb39ff6-d9df-4c40-8137-339f7dd8f684,Houston,Texas,overture,residential,RES1-1SNB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,4.0,nsi_max_2am_2pm_emp_students,481576706021005,204.425958,"POLYGON ((-95.50362 29.59614, -95.50369 29.596..."
97,ovt_03489d5d-321a-40ec-a666-d16c3e62fbb5,Houston,Texas,overture,residential,RES1-1SNB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,2.0,nsi_max_2am_2pm_emp_students,481576706021005,203.727220,"POLYGON ((-95.50348 29.59626, -95.50352 29.596..."
98,ovt_da476c1c-0324-4dba-98ff-46223d5b9607,Houston,Texas,overture,residential,RES1-1SNB,nsi_occtype,1.0,inferred_single_family,1.0,nsi,2.0,nsi_max_2am_2pm_emp_students,481576706021005,180.614207,"POLYGON ((-95.5036 29.59637, -95.50358 29.5964..."
